# Agentic AI: From Chatbots to Agents
**Presented by:** Sharad Rajore | **Organization:** Zensar Technologies

---

### 🎯 Learning Objectives
1. **Witness the Limitations:** See where a raw LLM fails (Math & Real-time data).
2. **Build Tools:** Create "Hands" for the AI using Python functions.
3. **The ReAct Loop:** Observe the *Thought -> Action -> Observation* cycle in real-time.

### 🛠️ Prerequisites
- Python 3.10+
- LangChain installed (`pip install langchain langchain-openai`)
- OpenAI API Key (or similar)

In [23]:
# 1. Setup Environment
from dotenv import load_dotenv
load_dotenv()

True

## Part 1: The "Brain in a Jar" (LLM Limitations)
First, let's look at a standard LLM. It is powerful but isolated. It cannot access the outside world and often hallucinates on complex math.

In [24]:
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Initialize the "Brain" (LLM)
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# TEST 1: The Math Trap
# LLMs predict the next word, they don't calculate. 
# For large numbers, they often guess.
query = "What is 49,823 multiplied by 19,283?"

response = llm.invoke(query)
print(f"User: {query}")
print(f"LLM Answer: {response.content}")

# Reality Check: 49823 * 19283 = 960,736,909
# Check if the LLM got it right (It often fails or is slightly off without a tool).

User: What is 49,823 multiplied by 19,283?
LLM Answer: The product is  

\[
49,\!823 \times 19,\!283 = 960,\!736,\!909.
\]


## Part 2: Giving the Brain "Hands" (Defining Tools)
To fix this, we don't train the model more. We give it a **Tool**. 
In LangChain, a tool is just a Python function with a `@tool` decorator.

In [25]:
from langchain_core.tools import tool

# 1. Define the Calculator Tool
@tool
def multiply(a: int, b: int) -> int:
    """ Multiplies two integers and returns the result."""
    return a * b

# 2. Define a "Mock" Search Tool (Simulating Real-time data)
@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol. 
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    # In a real agent, this would call Yahoo Finance API
    ticker_upper = ticker.upper().strip()
    if "ZENSAR" in ticker_upper:
        return "464.00 INR"
    elif "GOOGL" in ticker_upper or "GOOGLE" in ticker_upper:
        return "175.00 USD"
    else:
        return f"Unknown Ticker: {ticker}"

# List of tools available to our Agent
#tools = [multiply, get_stock_price]

print("Tools Created: Calculator (Hands) & Stock Search (Eyes)")

Tools Created: Calculator (Hands) & Stock Search (Eyes)


In [26]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults()

In [28]:
response = search_tool.invoke("what about US and Iran deal? is it finally done?")

In [29]:
response

"snippet: Middle East US-Iran Deal Finalised: What's Agreed, What's Still Pending Edited by: Apoorva Shukla Updated Jun 15, 2026, 07:17 IST US, Iran agree on framework peace deal to end war, reopen Hormuz and lift blockade, but implementation awaits Friday signing, with nuclear issue unresolved and further talks planned over 60 days., title: US-Iran Deal Finalised: What's Agreed, What's Still Pending, link: https://www.timesnownews.com/world/middle-east/us-iran-deal-finalised-whats-agreed-whats-still-pending-explained-article-154623406, snippet: Are Iran, US really close to a breakthrough 'deal'? Trump says a deal with Iran could be signed in Europe over the weekend. Tehran cautions against speculation., title: Are Iran, US really close to a breakthrough 'deal'?, link: https://www.aljazeera.com/features/2026/6/12/are-iran-us-really-close-to-a-breakthrough-deal, snippet: The U.S. and Iran agreed on a peace deal to bring immediate and permanent termination to the conflict. The deal follo

## Part 3: The Agent (ReAct Loop)
Now we bind the tools to the LLM. The LLM will now act as a router:
1. **Thought:** Do I know the answer? No.
2. **Action:** Pick the right tool.
3. **Observation:** Get the output.
4. **Final Answer:** Summarize for the user.

In [6]:
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Initialize the "Brain" (LLM)
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

In [ ]:
from langchain.agents import create_agent

# Create a ReAct agent using the modern LangChain API
# debug=True enables verbose output showing the Thought -> Action -> Observation loop
#agent_executor = create_agent(llm, tools, debug=True)
agent_executor = create_agent(
    model=llm,
    tools=[multiply, get_stock_price,search_tool],
    system_prompt="You are a helpful assistant you only provide the answers based on the tools ,don't use your own knowledge and don't make up any answer if you don't have the answer in the tools, if you don't have the answer in the tools then say I don't know",
)


print("Agent created successfully with ReAct capabilities!")

NameError: name 'multiply' is not defined

## Part 4: Agent in Action
Let's ask the exact same math question, plus a question about Zensar's stock.

In [ ]:
print("--- SCENARIO 1: MATH ---")
result1 = agent_executor.invoke({"messages": [("user", "What is 49,823 multiplied by 19,283?")]})
print(f"\nFinal Answer: {result1['messages'][-1].content}")
#result1



--- SCENARIO 1: MATH ---

Final Answer: The result of multiplying 49,823 by 19,283 is 960736909.


In [ ]:
print("\n\n--- SCENARIO 2: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "What is the current stock price of Zensar?")]})
print(f"\nFinal Answer: {result2['messages'][-1].content}")



--- SCENARIO 2: REAL-TIME DATA ---

Final Answer: The current stock price of Zensar is ₹464.00 (one thousand four hundred sixty-four) Indian Rupees.


In [ ]:



print("\n\n--- SCENARIO 3: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "What is the current stock price of Zensar then after that could you help me with What is 49823 * 19283? then tell me the capital of India")]})
#print(f"\nFinal Answer: {result2['messages'][-1].content}")
result2



--- SCENARIO 3: REAL-TIME DATA ---


{'messages': [HumanMessage(content='What is the current stock price of Zensar then after that could you help me with What is 49823 * 19283? then tell me the capital of India', additional_kwargs={}, response_metadata={}, id='9b66c7ae-afe4-4dd9-9e06-ce91855bae4d'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-06-18T10:41:37.018682Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11758111800, 'load_duration': 936756100, 'prompt_eval_count': 312, 'prompt_eval_duration': 3115304000, 'eval_count': 78, 'eval_duration': 7668317000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019eda52-4508-79d0-b6bf-168316f4e8da-0', tool_calls=[{'name': 'get_stock_price', 'args': {'ticker': 'ZENSAR'}, 'id': '8645d080-f3bb-4d56-91b4-40d6664e55e6', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'b': 19283, 'a': 49823}, 'id': '5bcc5bd7-8103-480c-b2d0-0f83cf77d451', 'type': 'tool_call'}, {'name'

### 🧠 What just happened?
1. **Math:** Instead of guessing, you should see the agent say `Invoking: multiply`. It used the calculator.
2. **Stock:** Instead of saying "I don't know", it said `Invoking: get_stock_price`. 

**This is the shift from Generative AI (guessing) to Agentic AI (executing).**

In [33]:
print("\n\n--- SCENARIO 4: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "In India vs Afghanistan cricket match yesterday 17th June 2026, how many runs India has scored?")]})
#print(f"\nFinal Answer: {result2['messages'][-1].content}")
result2



--- SCENARIO 4: REAL-TIME DATA ---


{'messages': [HumanMessage(content='In India vs Afghanistan cricket match yesterday 17th June 2026, how many runs India has scored?', additional_kwargs={}, response_metadata={}, id='2ed415af-6af4-45fe-9b57-862f558222a1'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-18T11:28:26.517799008Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1202373611, 'load_duration': None, 'prompt_eval_count': 307, 'prompt_eval_duration': None, 'eval_count': 111, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019eda7d-43c3-7603-b9b5-87dbebdda5d8-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'India vs Afghanistan cricket match 17 June 2026 runs India scored'}, 'id': '2c604586-22fc-427e-98b3-7eaf426b8efe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 307, 'output_tokens': 111, 'total_tokens': 418}),
  ToolMessag

In [9]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key="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")
response = tavily_client.search(query="Who is Leo Messi?",max_results=3)

print(response)

{'query': 'Who is Leo Messi?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Lionel_Messi', 'title': 'Lionel Messi - Wikipedia', 'content': '**Lionel Andrés** "**Leo**" **Messi** (born 24 June 1987) is an Argentine professional footballer who plays as a forward "Forward (association football)") for and captains "Captain (association football)") both the Major League Soccer club Inter Miami and the Argentina national team. In 2025, he was named the All Time Men\'s World Best Player by the IFFHS. His records include most goals in a calendar year (91), most goals for a single club (672 for Barcelona), most goals in La Liga (474), most assists#Players_with_most_assists_in_international_football_(all-time) "Assist (association football)") in international football (61), most goal contributions in the FIFA World Cup (21), most goals scored in the FIFA World Cup (16, alongside Miroslav Klose), and most goal contributions in the C